# Visual Storytelling & Optimización de Flota Divvy Chicago
### Análisis de Movilidad Urbana, POO y Vectorización Científica

**Autor:** Luis Monge  
**Metodología:** CRISP-DM  
**Dataset Oficial:** Divvy Trip Data (Chicago Open Data Portal / AWS S3)

## 1. Carga Adaptativa e Ingesta de Datos

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset
from src.processor import MobilityCleaner, MobilityAnalyst
from src.optimization import benchmark_ejecucion_bucle, benchmark_ejecucion_vectorizada, prueba_hipotesis_duracion_usuarios

df_raw = load_dataset(force_sample=False, nrows=5000) # Carga adaptativa
cleaner = MobilityCleaner()
df_limpio = cleaner.process(df_raw)
print(f"Métricas del limpiador POO: {cleaner.stats}")

## 2. Benchmarking de Rendimiento: Bucle vs Vectorización NumPy

In [2]:
df_eval = df_limpio.head(1000)
_, metricas_bucle = benchmark_ejecucion_bucle(df_eval)
_, metricas_vec = benchmark_ejecucion_vectorizada(df_eval)

aceleracion = metricas_bucle['tiempo_segundos'] / max(1e-6, metricas_vec['tiempo_segundos'])
print(f"Tiempo bucle iterativo: {metricas_bucle['tiempo_segundos']:.5f}s")
print(f"Tiempo vectorizado NumPy: {metricas_vec['tiempo_segundos']:.5f}s")
print(f"Factor de aceleración: {aceleracion:.1f}x más rápido")

## 3. Pruebas de Hipótesis Estadísticas (Miembros vs Ocasionales)

In [3]:
res_hipotesis = prueba_hipotesis_duracion_usuarios(df_limpio)
for k, v in res_hipotesis.items():
    print(f"- {k}: {v}")

## 4. Visual Storytelling (Maximizando Ratio Data-Ink)

### Viz 1: Mapa de Calor Despined (Día de la Semana vs Hora)

In [4]:
pivote = df_limpio.pivot_table(index='day_of_week', columns='hour', values='ride_id', aggfunc='count', fill_value=0)
dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivote = pivote.reindex([d for d in dias if d in pivote.index])

fig, ax = plt.subplots(figsize=(12, 5), dpi=120)
sns.heatmap(pivote, cmap='viridis', cbar_kws={'label': 'Volumen de Viajes'}, linewidths=0.5, ax=ax)
sns.despine(top=True, right=True, left=False, bottom=False)
ax.set_title("Demanda Horaria de Viajes Divvy Chicago (Día vs Hora)", fontsize=14, weight='bold', pad=15)
ax.set_xlabel("Hora del Día (Ciclo 24 Horas)", fontsize=11)
ax.set_ylabel("Día de la Semana", fontsize=11)
plt.tight_layout()
plt.show()